<h1 align="center">❂ Wolf-Rayet Search ❂</h1>
<h3 align="center">Taller de investigación | Research workshop</h3>
<h4 align="center">Universidad de la Serena</h4>

---

**👨‍🎓 Student:** Diego Miranda

---

### 


In [ ]:
### Librarys ###
### Standar ###
import pandas as pd
import numpy as np
import math
import time, os, sys, gc
from pathlib import Path
from IPython.display import display


### Visualization ###
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
ROOT_DIR = Path().resolve()
GRAPHS_DIR = ROOT_DIR / 'graphs'
DATA_DIR = ROOT_DIR / 'data'
UTILS = ROOT_DIR / 'src' / 'modeling'
QDIR = DATA_DIR / "query_results"

pd.set_option('display.max_columns', None)

In [21]:
### We import the latest GWRC catalogue (v1.32, Jul 2025) ###
tablas = pd.read_html(DATA_DIR / 'GWRC.html')  
df = max(tablas, key=lambda t: t.shape[0])
df.to_csv(DATA_DIR / "GWRC.csv", index=False)

wolf_rayet = pd.read_csv(DATA_DIR / "GWRC.csv")
print(f'Number of Wolf-Rayet stars: {len(wolf_rayet)}' )
display(wolf_rayet.head())
display(wolf_rayet.describe)
wolf_rayet.dtypes

Number of Wolf-Rayet stars: 710


,ID,WR#,Reference,HD,Alias1,Alias2,Alias3,Right Ascension J2000,Declination J2000,Galactic Longitude (deg),Galactic Latitude (deg),Spectral Type,Spectral Type Reference,Binary Status,Binary Status Reference,u (WR),b (WR),v (WR),r (WR),U,B,V,G,J,H,K,Cluster,Association,Star Forming Region,Distance (kpc),Distance Reference,Nebula
0,1,1,VII,HD 4004,DR3 524101256981955200,NaN,NaN,00 43 28.39,+64 45 35.4,122.0825,1.9012,WN4b,SSM96,SB1?,"La83, MS86, MMH98, MG99, Ni00",11.20,11.02,10.51,10.15,10.36,10.74,10.10,9.79,8.206,7.857,7.48,NaN,(Cas OB7),NaN,3.015,CRB23 (DR3),NaN
1,2,2,VII,HD 6327,DR3 426378793809476096,NaN,NaN,01 05 23.03,+60 25 18.9,124.6536,-2.4032,WN2b,SSM96,VB,"Hip97, MMH98",11.62,11.46,11.33,11.17,11.04,10.35,10.99,11.04,10.036,9.783,9.445,NaN,Cas OB1:,NaN,2.4,CSM19,NaN
2,3,3,VII,HD 9974,DR3 508955656105078912,NaN,NaN,01 38 55.62,+58 09 22.6,129.1797,-4.1382,WN3ha,MMC04,SB2,"MLS86,MS86,SH89,SM89,SS96",10.54,10.64,10.7,10.73,9.85,10.62,10.61,10.58,10.237,10.134,10.009,NaN,NaN,NaN,2.215,CRB23,NaN
3,773,3-1,MMC25,NaN,DR3 512856654641310976,WR-C-01,NaN,01 40 32.96,+63 42 22.9,128.3387,1.3514,WN4,MMC05,NaN,NaN,NaN,16.04,14.96,NaN,NaN,NaN,NaN,13.89,11.56,11.06,10.62,NaN,NaN,NaN,6.11,MMC25,NaN
4,4,4,VII,HD 16523,DR3 454818035714964992,NaN,NaN,02 41 11.67,+56 43 49.8,137.5948,-2.9839,WC5+?,VI,"SB1, no d.e.l.","MS86,CM89,RC89,SS90,VII",10.73,10.73,10.53,10.53,10.03,10.30,9.80,9.68,8.749,8.566,7.877,NaN,NaN,NaN,2.632,CRB23,NaN


<bound method NDFrame.describe of       ID  WR# Reference         HD                   Alias1   Alias2  \
0      1    1       VII    HD 4004   DR3 524101256981955200      NaN   
1      2    2       VII    HD 6327   DR3 426378793809476096      NaN   
2      3    3       VII    HD 9974   DR3 508955656105078912      NaN   
3    773  3-1     MMC25        NaN   DR3 512856654641310976  WR-C-01   
4      4    4       VII   HD 16523   DR3 454818035714964992      NaN   
..   ...  ...       ...        ...                      ...      ...   
705  429  155       VII  HD 214419  DR3 2006888825592397440   CQ Cep   
706  430  156       VII        NaN  DR3 2014674226896884864   MR 119   
707  431  157       VII  HD 219460  DR3 2013906561629759232      NaN   
708  432  158       VII        NaN  DR3 2012912500041834752   MR 112   
709  433  159      VIIA        NaN  DR3 2016055591466867712    BCC 1   

           Alias3 Right Ascension J2000 Declination J2000  \
0             NaN           00 43 28.39 

ID                            int64
WR#                          object
Reference                    object
HD                           object
Alias1                       object
Alias2                       object
Alias3                       object
Right Ascension J2000        object
Declination J2000            object
Galactic Longitude (deg)    float64
Galactic Latitude (deg)     float64
Spectral Type                object
Spectral Type Reference      object
Binary Status                object
Binary Status Reference      object
u (WR)                      float64
b (WR)                       object
v (WR)                       object
r (WR)                      float64
U                           float64
B                           float64
V                           float64
G                           float64
J                            object
H                            object
K                            object
Cluster                      object
Association                 

In [ ]:
# --- Cleaning Format ------------------------------------
### Columns names ###
df = wolf_rayet.copy()
df.columns = (
    df.columns
    .str.strip()
)

print(f'Columns before filters: {df.columns} \nShape before filters: {df.shape}')

### Select Only Gaia Identifiers ###
df['Alias1'] = df['Alias1'].astype(str).str.strip()
cond_gaia_match = df['Alias1'].str.match(r'DR\d\s+\d',na=False)
df = df[cond_gaia_match]
df['Alias1'] = 'Gaia ' + df['Alias1']

### Select Columns ####
selected_columns = [
    'WR#', 'Alias1', 'Right Ascension J2000', 'Declination J2000'
]

df = df[[col for col in selected_columns]]

print(f'Columns after filters: {df.columns} \nShape after filters: {df.shape}')
duplicated = df['Alias1'].duplicated()
print(f'Duplicated identifiers: {duplicated.sum()}')
display(df.head())
df.to_csv(DATA_DIR / 'wr_from_GWRC_cleaned.csv')

del df, cond_gaia_match, selected_columns
gc.collect()

Columns before filters: Index(['ID', 'WR#', 'Reference', 'HD', 'Alias1', 'Alias2', 'Alias3',
       'Right Ascension J2000', 'Declination J2000',
       'Galactic Longitude (deg)', 'Galactic Latitude (deg)', 'Spectral Type',
       'Spectral Type Reference', 'Binary Status', 'Binary Status Reference',
       'u (WR)', 'b (WR)', 'v (WR)', 'r (WR)', 'U', 'B', 'V', 'G', 'J', 'H',
       'K', 'Cluster', 'Association', 'Star Forming Region', 'Distance (kpc)',
       'Distance Reference', 'Nebula'],
      dtype='object') 
Shape before filters: (710, 32)
Columns after filters: Index(['WR#', 'Alias1', 'Right Ascension J2000', 'Declination J2000'], dtype='object') 
Shape after filters: (441, 4)
Duplicated identifiers: 0


,WR#,Alias1,Right Ascension J2000,Declination J2000
0,1,Gaia DR3 524101256981955200,00 43 28.39,+64 45 35.4
1,2,Gaia DR3 426378793809476096,01 05 23.03,+60 25 18.9
2,3,Gaia DR3 508955656105078912,01 38 55.62,+58 09 22.6
3,3-1,Gaia DR3 512856654641310976,01 40 32.96,+63 42 22.9
4,4,Gaia DR3 454818035714964992,02 41 11.67,+56 43 49.8


571

In [ ]:
# ------- Obtaining Gaia, Wise and 2MASS data -------------
# ------- We use querys from src/querys/gaia.py -----------

from src.querys.gaia import wr_from_gaia
wolf_rayet = pd.read_csv(DATA_DIR / 'wr_from_GWRC_cleaned.csv')

gaia_wr, not_found_core = wr_from_gaia(
    wolf_rayet,
    identifier_col="Alias1",
    df_not_found=True,
    output_route=DATA_DIR / "query_results/wr_gaia.csv", 
)

not_found_core.to_csv(QDIR/ "wr_not_found_gaia.csv", index = False)
print(f"Matched with Gaia (Gaia+2MASS+WISE): {gaia_wr.shape[0]}")
print(f"Matched columns: {gaia_wr.columns}")
print(f"Core-only (no matches found): {not_found_core.shape[0]}")

### Note: Since Gaia DR3 does not include all possible crossmatches with WISE and 2MASS, ###
### we will perform additional searches directly in the 2MASS and WISE catalogues. ###
### We will use a 5-arcsecond radius, or 1 arcsecond for sources with multiple candidates. ###

Launching Gaia crossmatch query (Gaia + 2MASS + WISE)...
INFO: Query finished. [astroquery.utils.tap.core]
Gaia crossmatch query completed: 274 rows.
Launching Gaia-core-only query for 167 missing identifiers...
INFO: Query finished. [astroquery.utils.tap.core]
Gaia-core-only query completed: 167 rows.
Matched with Gaia (Gaia+2MASS+WISE): 274
Matched columns: Index(['source_id', 'pmRA', 'e_pmDE', 'ra', 'parallax', 'radial_velocity',
       'e_Gflux', 'parallax_over_error', 'BPflux', 'teff', 'e_radial_velocity',
       'BPmag', 'logg', 'ruwe', 'RPflux', 'e_pmRA', 'Gmag', 'gaia_id', 'RPmag',
       'feh', 'e_RPflux', 'pmDE', 'e_BPflux', 'e_parallax', 'Gflux', 'dec',
       'Jmag', 'Kmag', 'qual_tmass', 'e_Kmag', 'e_Hmag', 'tmass_id', 'Hmag',
       'e_Jmag', 'tmass_oid', 'e_W4mag', 'e_W3mag', 'e_W2mag', 'W1mag',
       'W2mag', 'wise_id', 'e_W1mag', 'W4mag', 'qual_wise', 'W3mag',
       'allwise_oid', 'Alias1'],
      dtype='object')
Core-only (no matches found): 167


In [3]:
# ------- Obtaining Gaia, Wise and 2MASS data from not found before -------------
# ------- We use querys from src/querys/other.py --------------------------------

from src.querys.others import query_2mass_nf_bulk, query_wise_nf_bulk

gaia_nf = pd.read_csv(QDIR / "wr_not_found_gaia.csv")  

tmass_raw = query_2mass_nf_bulk(
    gaia_nf,
    radius_arcsec=5.0,
    save_csv=QDIR / "wr_2mass_nf_raw.csv",   
    chunk_size=None, 
)

wise_raw = query_wise_nf_bulk(
    gaia_nf,
    radius_arcsec=5.0,
    save_csv=QDIR / "wr_wise_nf_raw.csv",
    chunk_size=None,
)

print(len(tmass_raw), len(wise_raw))
print(f'2MASS columns names: {tmass_raw.columns}')
print(f'WISE columns names: {wise_raw.columns}')

194 133
2MASS columns names: Index(['_r', '2MASS', 'RAJ2000', 'DEJ2000', 'Jmag', 'Hmag', 'Kmag', 'e_Jmag',
       'e_Hmag', 'e_Kmag', 'Qflg', 'query_idx', 'gaia_id', 'sep_arcsec'],
      dtype='object')
WISE columns names: Index(['_r', 'AllWISE', 'RAJ2000', 'DEJ2000', 'W1mag', 'W2mag', 'W3mag',
       'W4mag', 'e_W1mag', 'e_W2mag', 'e_W3mag', 'e_W4mag', 'qph', 'query_idx',
       'gaia_id', 'sep_arcsec'],
      dtype='object')


In [9]:
from src.querys.utils import (
    nf_select_duplicates_by_radius,
    normalize_catalog,
    unify_gaia_with_nf,
)
gaia_nf   = pd.read_csv(QDIR / "wr_not_found_gaia.csv")   
tmass_raw = pd.read_csv(QDIR / "wr_2mass_nf_raw.csv")   
wise_raw  = pd.read_csv(QDIR / "wr_wise_nf_raw.csv")    

### Eliminate multiple results in the querys ##
tmass_sel, s2 = nf_select_duplicates_by_radius(tmass_raw, primary_radius_arcsec=1.5, secondary_radius_arcsec=1.0)
wise_sel,  sW = nf_select_duplicates_by_radius(wise_raw,  primary_radius_arcsec=1.5, secondary_radius_arcsec=1.0)

print("2MASS stats:", s2)
print("WISE  stats:", sW)

tmass_sel.to_csv(QDIR / "wr_2mass_nf_selected.csv", index=False)
wise_sel.to_csv(QDIR / "wr_wise_nf_selected.csv", index=False)

### Unify with Gaia ###
merged = unify_gaia_with_nf(
    gaia_core_df=gaia_nf,
    tmass_nf_df=tmass_sel, 
    wise_nf_df=wise_sel,    
    keep_also=("gaia_id","Alias1"),  
    reorder=True,
    require_all_nf=True    
)

merged.to_csv(QDIR / "wr_nf_merged.csv", index=False)
print("Not Found WR:", len(gaia_nf), " | New Querys WR:", merged.shape[0])
print(f'NF WR Columns: {merged.columns.tolist()}')

2MASS stats: {'n_input': 194, 'n_groups': 156, 'n_singletons_kept': 121, 'n_dupe_groups': 35, 'n_resolved_at_1p5': 34, 'n_resolved_at_1p0': 0, 'n_dropped_ambiguous': 1, 'n_dropped_zero_after_1p5': 1, 'dropped_qids': [163]}
WISE  stats: {'n_input': 133, 'n_groups': 132, 'n_singletons_kept': 131, 'n_dupe_groups': 1, 'n_resolved_at_1p5': 0, 'n_resolved_at_1p0': 0, 'n_dropped_ambiguous': 1, 'n_dropped_zero_after_1p5': 1, 'dropped_qids': [54]}
Not Found WR: 167  | New Querys WR: 127
NF WR Columns: ['Alias1', 'gaia_id', 'source_id', 'ra', 'dec', 'Gmag', 'BPmag', 'RPmag', 'parallax', 'e_parallax', 'pmRA', 'e_pmRA', 'pmDE', 'e_pmDE', 'ruwe', 'teff', 'logg', 'feh', 'tmass_id', 'Jmag', 'Hmag', 'Kmag', 'e_Jmag', 'e_Hmag', 'e_Kmag', 'qual_tmass', 'wise_id', 'W1mag', 'W2mag', 'W3mag', 'W4mag', 'e_W1mag', 'e_W2mag', 'e_W3mag', 'e_W4mag', 'qual_wise', 'has_tmass_nf', 'has_wise_nf', 'radial_velocity', 'parallax_over_error', 'Gflux', 'e_Gflux', 'BPflux', 'e_BPflux', 'RPflux', 'e_RPflux', 'e_radial_velo

In [66]:
### Unifiying all queries ###
gaia_wr = pd.read_csv(QDIR / 'wr_gaia.csv')
gaia_nf = pd.read_csv(QDIR / 'wr_nf_merged.csv')

gaia_wr.drop(columns=['allwise_oid', 'tmass_oid'], inplace= True ) # Columns we will not use and are not in gaia_nf

# --- We will select the columns that we are going to work with. We descarted W3 and W4 from Wise, because they will reduce ---------------
# --- a lot our dataset (low quality photometry in general). ------------------------------------------------------------------------------
selected_columns = ['source_id', 'ra', 'dec', 'Gmag', 'BPmag', 'RPmag', 'parallax', 'parallax_over_error',
                    'ruwe', 'tmass_id', 'Jmag', 'Hmag', 'Kmag', 'qual_tmass', 'wise_id', 'W1mag', 'W2mag', 'qual_wise']


gaia_wr = gaia_wr[selected_columns]
gaia_nf = gaia_nf[selected_columns]

wr_complete = pd.concat([gaia_wr, gaia_nf], ignore_index=True).drop_duplicates()
print(f'Complete dataframe of WR Stars with Gaia, 2MASS and Wise photometry: {len(wr_complete)}')
print(f'Selected columns for the proyect: {wr_complete.columns}')
counts = wr_complete.groupby('source_id').size()
duplicated_ids = counts[counts > 1]
print(f'Number of duplicated ids: {duplicated_ids.sum()}')

### Filter for parallax error ### 
# wr_complete = wr_complete[(wr_complete['parallax_over_error'] > 3) & (wr_complete['parallax'] > 0)]
# print(f'WR good parallax {wr_complete.shape}')
### Deleting Nan rows ###
print(f'NaN counts per column: {wr_complete.isna().sum()}')
wr_complete = wr_complete.dropna(subset=selected_columns)
print(f'Final shape after deleting NaNs: {wr_complete.shape}')
gc.collect()

Complete dataframe of WR Stars with Gaia, 2MASS and Wise photometry: 401
Selected columns for the proyect: Index(['source_id', 'ra', 'dec', 'Gmag', 'BPmag', 'RPmag', 'parallax',
       'parallax_over_error', 'ruwe', 'tmass_id', 'Jmag', 'Hmag', 'Kmag',
       'qual_tmass', 'wise_id', 'W1mag', 'W2mag', 'qual_wise'],
      dtype='object')
Number of duplicated ids: 0
NaN counts per column: source_id               0
ra                      0
dec                     0
Gmag                    4
BPmag                  14
RPmag                  10
parallax               12
parallax_over_error    12
ruwe                   12
tmass_id                0
Jmag                    0
Hmag                    0
Kmag                    0
qual_tmass              0
wise_id                 0
W1mag                   1
W2mag                   1
qual_wise               0
dtype: int64
Final shape after deleting NaNs: (378, 18)


68

In [67]:
### Selecting photometry quality ###
mask_tmass = wr_complete['qual_tmass'].str.startswith('AAA', na=False)
mask_wise  = wr_complete['qual_wise'].str.startswith('AA', na=False)

cond = wr_complete[mask_tmass & mask_wise]

print(f"AAA rows in 2MASS: {mask_tmass.sum()}")
print(f"AA rows in WISE: {mask_wise.sum()}")
print(f"Both condition rows: {len(cond)}")

wr_complete.to_csv(QDIR/ 'wr_fullsample_no_cleaned.csv', index=False)
cond.to_csv(QDIR/ 'wr_fullsample_cleaned.csv', index=False)

AAA rows in 2MASS: 348
AA rows in WISE: 323
Both condition rows: 299
